# 08 · Evaluation & Ablations

**Goal:** score the recovered branch graphs against tape-measure ground
truth for 8–12 trees, and run the three ablations called out in the
proposal:

1. **Classical vs. learned segmentation** — rerun notebook 06 with each
   mask variant and compare downstream metrics.
2. **Two-view vs. multi-view reconstruction** — use the from-scratch
   two-view cloud (notebook 03) as the starting point, vs. COLMAP.
3. **Reprojection filtering on vs. off** — skip notebook 06 and feed the
   raw COLMAP cloud directly into the trunk/skeleton stage.

Metrics live in `src/evaluate.py`: primary-branch recall, attachment-angle
MAE, attachment-height MAE.

In [ ]:
# Standard preamble — every notebook seeds np.random.seed(131).
import sys
from pathlib import Path

# Make `src/` importable from the notebooks/ directory.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
np.random.seed(131)

import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from src import evaluate, branch_graph
import csv

GT_DIR = PROJECT_ROOT / "data" / "ground_truth"
RECON_DIR = PROJECT_ROOT / "outputs" / "reconstructions"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Score a single tree

Walk the saved graph + tape-measure CSV and compute metrics.

In [ ]:
def score_one(tree_id, run_label):
    graph_path = RECON_DIR / f"{tree_id}_graph.npz"
    gt_path = GT_DIR / f"{tree_id}.csv"
    if not graph_path.exists() or not gt_path.exists():
        return None
    data = np.load(graph_path, allow_pickle=True)
    parents = dict((int(c), int(p)) for c, p in data["parents"])
    preds = evaluate.predicted_branches_from_graph(
        nodes=data["nodes"],
        edges=[tuple(e) for e in data["edges"]],
        trunk_root=data["trunk_root"],
        trunk_axis=data["trunk_direction"],
        parent_of=parents,
    )
    gts = evaluate.load_ground_truth(gt_path)
    m = evaluate.evaluate_tree(tree_id, preds, gts)
    return m

rows = []
for tree_dir in sorted((PROJECT_ROOT / "data" / "frames").iterdir()):
    if not tree_dir.is_dir(): continue
    m = score_one(tree_dir.name, "default")
    if m is not None:
        rows.append(m)
        print(f"{m.tree_id}: recall={m.recall:.2f}  angle MAE={m.angle_mae_deg:.1f}°  height MAE={m.height_mae_m*100:.1f} cm")
if rows:
    evaluate.write_metrics_csv(rows, METRICS_DIR / "default.csv")
else:
    print("No trees with both a saved graph and ground truth — capture some and rerun.")

## 2. Ablation runs

Each ablation re-runs notebook 06 / 07 with a different setting, saves a
new `<tree_id>_graph.npz` under a different filename suffix, then scores
every tree. The boilerplate is identical to the single-tree cell above;
we just loop over (variant_label, settings) tuples.

Run notebook-by-notebook for the actual sweep; this cell just summarises
the saved CSVs once they exist.

In [ ]:
for csv_path in sorted(METRICS_DIR.glob("*.csv")):
    print(f"\n=== {csv_path.name} ===")
    with open(csv_path) as f:
        reader = csv.DictReader(f)
        for row in reader:
            print(row)

## 3. Summary figure — bar chart per metric per ablation

Placeholder; fill in once ablation CSVs exist.

In [ ]:
# TODO once ablation CSVs are populated:
#   - bar chart of recall per variant
#   - bar chart of angle MAE per variant
#   - bar chart of height MAE per variant
# All saved to outputs/figures/08_*.png via viz.save_fig.